# Debug Pipeline — Analytical Fill

Interactive notebook version of `scripts/debug_pipeline.py`, using the
**analytical (point-in-polygon) renderer** instead of matplotlib
rasterisation.

Traces the full polygrid → atlas → globe pipeline step-by-step for
selected tiles with annotated diagnostic plots at each stage:

| Stage | What it shows |
|-------|---------------|
| **1. Globe topology** | All faces on the polyhedron with IDs, pentagon markers |
| **2. Detail grid** | Tutte embedding with boundary, macro-edge segments, detected corners |
| **3. Stitched composite** | Centre tile + neighbour aprons with view-limits box |
| **4. Corner matching** | Grid corners ↔ UV corners with angular alignment |
| **5. Sector equalisation** | Before/after grid corner adjustment with sector angles |
| **6. Warp sectors** | Triangle-fan decomposition with per-sector anisotropy |
| **7. Analytical fill** | Stitched tile rendered via point-in-polygon (no matplotlib raster) |
| **8. Warped tile** | Final warped slot with UV polygon boundary overlay |
| **9. Coverage diagnostics** | Pixel coverage, sentinel count, fill timing |

In [ ]:
# ── Setup & configuration ──────────────────────────────────────
import sys
import math
import time
import colorsys
from pathlib import Path

ROOT = Path("/home/toms/git_repos/pgrid")
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
from PIL import Image

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 120

# ── Parameters (edit these) ────────────────────────────────────
FREQ = 3
DETAIL_RINGS = 4
TILE_SIZE = 512
GUTTER = 4
SEED = 42

# Tiles to debug — None = auto-select (1 pent + 2 hex)
DEBUG_TILES = None
# DEBUG_TILES = ["t0", "t5", "t11"]

print("Setup OK")

In [ ]:
# ── Build globe + detail grids ─────────────────────────────────
from polygrid.globe import build_globe_grid
from polygrid.tile_detail import (
    TileDetailSpec, DetailGridCollection, build_tile_with_neighbours,
)
from polygrid.tile_uv_align import (
    get_macro_edge_corners,
    match_grid_corners_to_uv,
    compute_tile_view_limits,
    compute_grid_to_px_affine,
    warp_tile_to_uv,
    uv_polygon_px,
    _equalise_sector_ratios,
    _scale_corners_from_centroid,
    _build_sector_affines,
    _PENTAGON_GRID_SCALE,
)
from polygrid.uv_texture import get_tile_uv_vertices

print(f"Building globe (freq={FREQ})...")
grid = build_globe_grid(FREQ)
face_ids = sorted(grid.faces.keys(), key=lambda f: int(f[1:]))

spec = TileDetailSpec(detail_rings=DETAIL_RINGS)
print(f"Building detail grids (rings={DETAIL_RINGS})...")
t0 = time.perf_counter()
coll = DetailGridCollection.build(grid, spec)
print(f"  → {coll.total_face_count} sub-faces in {time.perf_counter()-t0:.2f}s")

# Select tiles to debug
if DEBUG_TILES:
    debug_ids = [t for t in DEBUG_TILES if t in face_ids]
else:
    pents = [f for f in face_ids if len(grid.faces[f].vertex_ids) == 5]
    hexes = [f for f in face_ids if len(grid.faces[f].vertex_ids) == 6]
    debug_ids = pents[:1] + hexes[:2]

print(f"\nDebug tiles: {debug_ids}")
for fid in debug_ids:
    n = len(grid.faces[fid].vertex_ids)
    print(f"  {fid}: {'pentagon ★' if n == 5 else 'hexagon'}")

# Tile hues (golden-ratio spread)
golden = 0.618033988749895
tile_hues = {fid: (0.08 + i * golden) % 1.0 for i, fid in enumerate(face_ids)}

## Stage 1 — Globe Topology

Azimuthal equidistant projection of all Goldberg faces. Pentagons
marked with ★, debug tiles highlighted.

In [ ]:
# ── Stage 1: Globe topology ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_title("Stage 1: Globe Topology", fontsize=14, fontweight="bold")
ax.set_aspect("equal")
ax.axis("off")

patches, colours = [], []
for fid in face_ids:
    face = grid.faces[fid]
    verts_3d = [(grid.vertices[v].x, grid.vertices[v].y, grid.vertices[v].z)
                for v in face.vertex_ids]
    # Azimuthal equidistant projection from +Z
    verts_2d = []
    for x, y, z in verts_3d:
        r3 = math.sqrt(x*x + y*y + z*z)
        if r3 < 1e-12:
            verts_2d.append((0.0, 0.0)); continue
        theta = math.acos(max(-1, min(1, z / r3)))
        phi = math.atan2(y, x)
        verts_2d.append((theta * math.cos(phi), theta * math.sin(phi)))

    if len(verts_2d) >= 3:
        n_sides = len(face.vertex_ids)
        hue = tile_hues[fid]
        light = 0.55 if n_sides == 5 else 0.65
        # Highlight debug tiles
        if fid in debug_ids:
            light = 0.45
        r, g, b = colorsys.hls_to_rgb(hue, light, 0.8)
        patches.append(MplPolygon(verts_2d, closed=True))
        colours.append((r, g, b))

        cx = sum(v[0] for v in verts_2d) / len(verts_2d)
        cy = sum(v[1] for v in verts_2d) / len(verts_2d)
        label = fid + (" ★" if n_sides == 5 else "")
        fontsize = 7 if fid in debug_ids else 5
        ax.annotate(
            label, (cx, cy), fontsize=fontsize, ha="center", va="center",
            fontweight="bold", color="white",
            bbox=dict(
                boxstyle="round,pad=0.15",
                fc="red" if fid in debug_ids else "black",
                alpha=0.7,
            ),
        )

pc = PatchCollection(patches, facecolors=colours, edgecolors="white", linewidths=0.5)
ax.add_collection(pc)
ax.autoscale_view()
plt.tight_layout()
plt.show()

## Stages 2–8 — Per-Tile Pipeline

For each debug tile, the cell below runs through the full pipeline:

1. **Detail grid** — boundary, macro edges, corners with angles
2. **Stitched composite** — centre + neighbours with view-limits box
3. **Corner matching** — grid corners ↔ UV corners
4. **Sector equalisation** — before/after with angle annotations
5. **Warp sectors** — triangle-fan with anisotropy values
6. **Analytical fill** — point-in-polygon stitched render (no matplotlib raster)
7. **Warped tile** — final slot with UV polygon overlay
8. **Coverage diagnostics** — pixel coverage, timing, sentinel count

In [ ]:
# ── Helper: analytical fill for a composite ────────────────────
from polygrid.geometry import face_center as _face_center

SENTINEL_BG = np.array([31, 31, 31], dtype=np.uint8)

def analytical_fill_composite(
    merged, face_colours, xlim, ylim, tile_size,
):
    """Render tile via point-in-polygon fill.  Returns (img, elapsed, coverage%)."""
    img = np.full((tile_size, tile_size, 3), SENTINEL_BG, dtype=np.uint8)
    xs = np.linspace(xlim[0], xlim[1], tile_size)
    ys = np.linspace(ylim[1], ylim[0], tile_size)  # row 0 = top
    px, py = np.meshgrid(xs, ys)
    assigned = np.zeros((tile_size, tile_size), dtype=bool)

    t0 = time.perf_counter()
    for fid, face in merged.faces.items():
        colour = face_colours.get(fid)
        if colour is None:
            continue
        verts = []
        for vid in face.vertex_ids:
            v = merged.vertices.get(vid)
            if v is None or not v.has_position():
                break
            verts.append((v.x, v.y))
        else:
            if len(verts) < 3:
                continue
        if len(verts) < 3:
            continue

        path = MplPath(verts + [verts[0]])
        vx = [p[0] for p in verts]
        vy = [p[1] for p in verts]
        col_mask = (xs >= min(vx)) & (xs <= max(vx))
        row_mask = (ys >= min(vy)) & (ys <= max(vy))
        if not col_mask.any() or not row_mask.any():
            continue

        ci = np.where(col_mask)[0]
        ri = np.where(row_mask)[0]
        sub_pts = np.column_stack([px[np.ix_(ri, ci)].ravel(),
                                   py[np.ix_(ri, ci)].ravel()])
        mask = path.contains_points(sub_pts)
        rows_idx, cols_idx = np.meshgrid(ri, ci, indexing="ij")
        rows_flat = rows_idx.ravel()[mask]
        cols_flat = cols_idx.ravel()[mask]

        rgb = np.array([int(colour[0]*255), int(colour[1]*255),
                        int(colour[2]*255)], dtype=np.uint8)
        img[rows_flat, cols_flat] = rgb
        assigned[rows_flat, cols_flat] = True

    elapsed = time.perf_counter() - t0
    coverage = 100.0 * assigned.sum() / (tile_size * tile_size)
    return img, elapsed, coverage

print("Analytical fill helper defined ✓")

In [ ]:
# ── Per-tile pipeline ──────────────────────────────────────────
results = {}  # face_id → {corners, uv_corners, img, warped, ...}

for tile_id in debug_ids:
    n_sides = len(grid.faces[tile_id].vertex_ids)
    tile_type = "pentagon ★" if n_sides == 5 else "hexagon"
    print(f"\n{'='*70}")
    print(f"  {tile_id} — {tile_type} ({n_sides} sides)")
    print(f"{'='*70}")

    dg, _ = coll.get(tile_id)

    # ── Stage 2: Detail grid ──────────────────────────────────
    corner_ids = dg.metadata.get("corner_vertex_ids")
    dg.compute_macro_edges(n_sides=n_sides, corner_ids=corner_ids)
    grid_corners = get_macro_edge_corners(dg, n_sides)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_title(f"Stage 2: Detail Grid — {tile_id} ({tile_type})",
                 fontsize=12, fontweight="bold")
    ax.set_aspect("equal")

    # Draw faces
    for fid, face in dg.faces.items():
        verts = [(dg.vertices[v].x, dg.vertices[v].y)
                 for v in face.vertex_ids
                 if dg.vertices.get(v) and dg.vertices[v].has_position()]
        if len(verts) >= 3:
            ax.add_patch(MplPolygon(verts, closed=True,
                                    fc="#e0e0e0", ec="#888888", lw=0.3))

    # Boundary edges
    for edge in dg.edges.values():
        if len(edge.face_ids) < 2:
            va, vb = dg.vertices[edge.vertex_ids[0]], dg.vertices[edge.vertex_ids[1]]
            ax.plot([va.x, vb.x], [va.y, vb.y], "r-", lw=1.2, alpha=0.6)

    # Macro edges
    me_cols = plt.cm.tab10(np.linspace(0, 1, max(n_sides, 1)))
    for me in dg.macro_edges:
        col = me_cols[me.id % len(me_cols)]
        for j in range(len(me.vertex_ids) - 1):
            va, vb = dg.vertices[me.vertex_ids[j]], dg.vertices[me.vertex_ids[j+1]]
            ax.plot([va.x, vb.x], [va.y, vb.y], "-", color=col, lw=2.5)
        mid = dg.vertices[me.vertex_ids[len(me.vertex_ids)//2]]
        ax.annotate(f"ME{me.id}", (mid.x, mid.y), fontsize=7,
                    fontweight="bold", color=col,
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.8))

    # Corners with angles
    gcx = sum(c[0] for c in grid_corners) / len(grid_corners)
    gcy = sum(c[1] for c in grid_corners) / len(grid_corners)
    for k, (cx, cy) in enumerate(grid_corners):
        ax.plot(cx, cy, "ko", ms=7, zorder=5)
        ax.plot(cx, cy, "o", color=me_cols[k], ms=5, zorder=6)
        ax.annotate(f"C{k}", (cx, cy), fontsize=8, fontweight="bold",
                    xytext=(6, 6), textcoords="offset points",
                    bbox=dict(boxstyle="round,pad=0.15", fc="yellow", alpha=0.9))
        angle = math.degrees(math.atan2(cy - gcy, cx - gcx))
        ax.annotate(f"{angle:.1f}°", (cx, cy), fontsize=5, color="gray",
                    xytext=(-5, -10), textcoords="offset points")
    ax.plot(gcx, gcy, "r+", ms=12, mew=2, zorder=5)
    ax.autoscale_view(); ax.margins(0.05)
    plt.tight_layout(); plt.show()

    # ── Stage 3: Stitched composite ───────────────────────────
    composite = build_tile_with_neighbours(coll, tile_id, grid)
    mg = composite.merged
    xlim, ylim = compute_tile_view_limits(composite, tile_id)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_title(f"Stage 3: Stitched Composite — {tile_id}",
                 fontsize=12, fontweight="bold")
    ax.set_aspect("equal")

    comp_names = list(composite.id_prefixes.keys())
    comp_hues = {n: (0.08 + i * golden) % 1.0 for i, n in enumerate(comp_names)}
    p_list, c_list = [], []
    for fid, face in mg.faces.items():
        verts = [(mg.vertices[v].x, mg.vertices[v].y)
                 for v in face.vertex_ids
                 if mg.vertices.get(v) and mg.vertices[v].has_position()]
        if len(verts) < 3:
            continue
        comp_name = next((n for n, pfx in composite.id_prefixes.items()
                          if fid.startswith(pfx)), None)
        if comp_name is None:
            continue
        is_center = (comp_name == tile_id)
        hue = comp_hues[comp_name]
        r, g, b = colorsys.hls_to_rgb(hue,
                                       0.6 if is_center else 0.4,
                                       0.8 if is_center else 0.5)
        p_list.append(MplPolygon(verts, closed=True))
        c_list.append((r, g, b))

    ax.add_collection(PatchCollection(p_list, facecolors=c_list,
                                      edgecolors="#00000030", linewidths=0.3))
    # View-limits box
    rx = [xlim[0], xlim[1], xlim[1], xlim[0], xlim[0]]
    ry = [ylim[0], ylim[0], ylim[1], ylim[1], ylim[0]]
    ax.plot(rx, ry, "r--", lw=2, label="view limits")

    for name, pfx in composite.id_prefixes.items():
        xs_c = [mg.vertices[v].x for fid, f in mg.faces.items()
                if fid.startswith(pfx)
                for v in f.vertex_ids
                if mg.vertices.get(v) and mg.vertices[v].has_position()]
        ys_c = [mg.vertices[v].y for fid, f in mg.faces.items()
                if fid.startswith(pfx)
                for v in f.vertex_ids
                if mg.vertices.get(v) and mg.vertices[v].has_position()]
        if xs_c:
            ax.annotate(
                name + (" (centre)" if name == tile_id else ""),
                (sum(xs_c)/len(xs_c), sum(ys_c)/len(ys_c)),
                fontsize=6, ha="center", va="center", fontweight="bold",
                color="white",
                bbox=dict(boxstyle="round,pad=0.15", fc="black", alpha=0.7))

    ax.legend(fontsize=7); ax.autoscale_view(); ax.margins(0.05)
    plt.tight_layout(); plt.show()

    # ── Stage 4: Corner matching ──────────────────────────────
    uv_corners = get_tile_uv_vertices(grid, tile_id)
    grid_corners_matched = match_grid_corners_to_uv(grid_corners, grid, tile_id)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Stage 4: Corner Matching — {tile_id} ({tile_type})",
                 fontsize=12, fontweight="bold")
    n = len(uv_corners)

    # Panel 1: Raw grid corners
    ax = axes[0]; ax.set_title("Grid Corners"); ax.set_aspect("equal")
    gc = np.array(grid_corners)
    gc_c = gc.mean(axis=0)
    for k in range(n):
        j = (k+1) % n
        ax.plot([gc[k,0], gc[j,0]], [gc[k,1], gc[j,1]], "b-", lw=1.5)
        ax.plot(gc[k,0], gc[k,1], "bo", ms=7)
        ax.annotate(f"C{k}", gc[k], fontsize=8, color="blue",
                    xytext=(5, 5), textcoords="offset points")
    ax.plot(*gc_c, "r+", ms=10, mew=2)

    # Panel 2: UV corners
    ax = axes[1]; ax.set_title("UV Corners"); ax.set_aspect("equal")
    uv = np.array(uv_corners)
    uv_c = uv.mean(axis=0)
    for k in range(n):
        j = (k+1) % n
        ax.plot([uv[k,0], uv[j,0]], [uv[k,1], uv[j,1]], "g-", lw=1.5)
        ax.plot(uv[k,0], uv[k,1], "go", ms=7)
        ax.annotate(f"UV{k}", uv[k], fontsize=8, color="green",
                    xytext=(5, 5), textcoords="offset points")
    ax.plot(*uv_c, "r+", ms=10, mew=2)

    # Panel 3: Matched overlay
    ax = axes[2]; ax.set_title("Matched (normalised)"); ax.set_aspect("equal")
    gc_m = np.array(grid_corners_matched)
    gc_mn = (gc_m - gc_m.min(0)) / np.maximum(gc_m.ptp(0), 1e-9)
    uv_n = (uv - uv.min(0)) / np.maximum(uv.ptp(0), 1e-9)
    for k in range(n):
        ax.plot(gc_mn[k,0], gc_mn[k,1], "bs", ms=7)
        ax.plot(uv_n[k,0], uv_n[k,1], "g^", ms=7)
        ax.annotate("", xy=uv_n[k], xytext=gc_mn[k],
                    arrowprops=dict(arrowstyle="->", color="red", lw=1, alpha=0.7))
    plt.tight_layout(); plt.show()

    # ── Stage 5: Sector equalisation ──────────────────────────
    grid_corners_eq, src_centroid = _equalise_sector_ratios(
        grid_corners_matched, uv_corners,
        tile_size=TILE_SIZE, gutter=GUTTER)
    if n_sides == 5:
        grid_corners_eq = _scale_corners_from_centroid(
            grid_corners_eq, _PENTAGON_GRID_SCALE)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"Stage 5: Sector Equalisation — {tile_id}",
                 fontsize=12, fontweight="bold")
    for panel, (corners, title) in enumerate([
        (grid_corners_matched, "Before"), (grid_corners_eq, "After"),
    ]):
        ax = axes[panel]; ax.set_title(title); ax.set_aspect("equal")
        pts = np.array(corners)
        c = pts.mean(axis=0)
        for k in range(n):
            j = (k+1) % n
            ax.plot([pts[k,0], pts[j,0]], [pts[k,1], pts[j,1]], "b-", lw=1.5)
            ax.plot([c[0], pts[k,0]], [c[1], pts[k,1]], "r--", lw=0.7, alpha=0.5)
            a0 = math.atan2(pts[k,1]-c[1], pts[k,0]-c[0])
            a1 = math.atan2(pts[j,1]-c[1], pts[j,0]-c[0])
            span = (a1 - a0) % (2*math.pi)
            mid_a = a0 + span/2
            r = np.linalg.norm(pts[k]-c) * 0.4
            ax.annotate(f"{math.degrees(span):.1f}°",
                        (c[0]+r*math.cos(mid_a), c[1]+r*math.sin(mid_a)),
                        fontsize=7, ha="center", color="darkred", fontweight="bold")
            ax.plot(pts[k,0], pts[k,1], "bo", ms=6)
            ax.annotate(f"C{k}", pts[k], fontsize=7, xytext=(5, 5),
                        textcoords="offset points")
        ax.plot(*c, "r+", ms=10, mew=2)
        if panel == 1 and src_centroid is not None:
            ax.plot(src_centroid[0], src_centroid[1], "gx", ms=8, mew=2)
        ax.margins(0.1)
    plt.tight_layout(); plt.show()

    # ── Stage 6: Warp sectors ─────────────────────────────────
    src = np.array(grid_corners_eq, dtype=np.float64)
    dst_uv = np.array(uv_corners, dtype=np.float64)
    src_c = src.mean(axis=0)
    dst_px = np.empty_like(dst_uv)
    for i in range(n):
        dst_px[i, 0] = GUTTER + dst_uv[i, 0] * TILE_SIZE
        dst_px[i, 1] = GUTTER + (1.0 - dst_uv[i, 1]) * TILE_SIZE
    dst_px_c = dst_px.mean(axis=0)
    dst_angles = np.arctan2(dst_px[:,1]-dst_px_c[1], dst_px[:,0]-dst_px_c[0])
    order = np.argsort(dst_angles)
    src_sorted, dst_sorted = src[order], dst_px[order]
    fwd_sectors = _build_sector_affines(src_sorted, src_c, dst_sorted, dst_px_c)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"Stage 6: Warp Sectors — {tile_id}",
                 fontsize=12, fontweight="bold")
    sec_cols = plt.cm.Set3(np.linspace(0, 1, n))
    for panel, (pts, cent, title) in enumerate([
        (src_sorted, src_c, "Source (Grid)"),
        (dst_sorted, dst_px_c, "Dest (Slot px)"),
    ]):
        ax = axes[panel]; ax.set_title(title); ax.set_aspect("equal")
        for k in range(n):
            j = (k+1) % n
            tri = MplPolygon([cent, pts[k], pts[j]], closed=True,
                             fc=sec_cols[k], ec="black", lw=1, alpha=0.5)
            ax.add_patch(tri)
            tcx = (cent[0]+pts[k,0]+pts[j,0])/3
            tcy = (cent[1]+pts[k,1]+pts[j,1])/3
            A = fwd_sectors[k][0]
            svs = np.linalg.svd(A, compute_uv=False)
            aniso = svs[0]/svs[1] if svs[1] > 1e-12 else float("inf")
            ax.annotate(f"S{k}\n{aniso:.2f}×", (tcx, tcy), fontsize=6,
                        ha="center", fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.8))
        for k in range(n):
            ax.plot(pts[k,0], pts[k,1], "ko", ms=5, zorder=5)
        ax.plot(*cent, "r+", ms=10, mew=2, zorder=5)
        ax.autoscale_view(); ax.margins(0.1)
    plt.tight_layout(); plt.show()

    # ── Stage 7: Analytical fill ──────────────────────────────
    # Build colour-debug colours for the composite
    comp_centroids, comp_max_dist = {}, {}
    for name, pfx in composite.id_prefixes.items():
        xs_v, ys_v = [], []
        for fid, face in mg.faces.items():
            if not fid.startswith(pfx):
                continue
            c = _face_center(mg.vertices, face)
            if c: xs_v.append(c[0]); ys_v.append(c[1])
        if xs_v:
            cx, cy = sum(xs_v)/len(xs_v), sum(ys_v)/len(ys_v)
            comp_centroids[name] = (cx, cy)
            comp_max_dist[name] = max(
                math.hypot(x-cx, y-cy) for x, y in zip(xs_v, ys_v)) or 1.0
        else:
            comp_centroids[name] = (0., 0.)
            comp_max_dist[name] = 1.0

    face_colours = {}
    for fid, face in mg.faces.items():
        comp_name = next((n for n, pfx in composite.id_prefixes.items()
                          if fid.startswith(pfx)), None)
        if comp_name is None:
            continue
        hue = tile_hues.get(comp_name, 0.0)
        ccx, ccy = comp_centroids[comp_name]
        c = _face_center(mg.vertices, face)
        fx, fy = c if c else (ccx, ccy)
        t = min(math.hypot(fx-ccx, fy-ccy) / comp_max_dist[comp_name], 1.0)
        light = 0.72 - 0.20*t
        sat = 0.65 + 0.15*t
        face_colours[fid] = colorsys.hls_to_rgb(hue, light, sat)

    img_ana, elapsed, coverage = analytical_fill_composite(
        mg, face_colours, xlim, ylim, TILE_SIZE)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_title(f"Stage 7: Analytical Fill — {tile_id}\n"
                 f"{elapsed:.3f}s, {coverage:.1f}% coverage",
                 fontsize=12, fontweight="bold")
    ax.imshow(img_ana)
    # View-limits box
    ax.plot([0, TILE_SIZE-1, TILE_SIZE-1, 0, 0],
            [0, 0, TILE_SIZE-1, TILE_SIZE-1, 0], "r--", lw=1, alpha=0.5)
    ax.axis("off")
    plt.tight_layout(); plt.show()

    # ── Stage 8: Warp to UV ───────────────────────────────────
    tile_img = Image.fromarray(img_ana)
    slot_size = TILE_SIZE + 2*GUTTER
    warped = warp_tile_to_uv(
        tile_img, xlim, ylim,
        compute_grid_to_px_affine(grid_corners_eq, uv_corners,
                                  tile_size=TILE_SIZE, gutter=GUTTER),
        slot_size,
        grid_corners=grid_corners_eq,
        uv_corners=uv_corners,
        tile_size=TILE_SIZE,
        gutter=GUTTER,
        src_centroid_override=src_centroid,
    )

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_title(f"Stage 8: Warped Tile — {tile_id}",
                 fontsize=12, fontweight="bold")
    ax.imshow(np.array(warped))

    uv_px = uv_polygon_px(uv_corners, TILE_SIZE, GUTTER)
    uv_px_closed = list(uv_px) + [uv_px[0]]
    ax.plot([p[0] for p in uv_px_closed], [p[1] for p in uv_px_closed],
            "r-", lw=2, label="UV polygon")
    for k, (ppx, ppy) in enumerate(uv_px):
        ax.plot(ppx, ppy, "ro", ms=5)
        ax.annotate(f"UV{k}", (ppx, ppy), fontsize=7, color="red",
                    fontweight="bold", xytext=(4, -8), textcoords="offset points")
    # Gutter boundary
    ax.plot([GUTTER, TILE_SIZE+GUTTER, TILE_SIZE+GUTTER, GUTTER, GUTTER],
            [GUTTER, GUTTER, TILE_SIZE+GUTTER, TILE_SIZE+GUTTER, GUTTER],
            "y--", lw=1, alpha=0.6, label="inner tile")
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xlim(0, slot_size); ax.set_ylim(slot_size, 0)
    plt.tight_layout(); plt.show()

    # ── Stage 9: Coverage diagnostics ─────────────────────────
    sentinel_mask = np.all(img_ana == SENTINEL_BG, axis=2)
    warped_arr = np.array(warped)
    warped_sentinel = np.all(warped_arr == SENTINEL_BG, axis=2)

    print(f"\n  ── {tile_id} diagnostics ──")
    print(f"    Analytical fill: {elapsed:.3f}s")
    print(f"    Pre-warp coverage: {coverage:.1f}% "
          f"({sentinel_mask.sum()} sentinel px)")
    print(f"    Post-warp sentinel: {warped_sentinel.sum()} px "
          f"({100*warped_sentinel.sum()/(slot_size*slot_size):.1f}%)")

    results[tile_id] = {
        "grid_corners": grid_corners,
        "uv_corners": uv_corners,
        "grid_corners_eq": grid_corners_eq,
        "img": img_ana,
        "warped": warped_arr,
        "elapsed": elapsed,
        "coverage": coverage,
    }

## Summary

Side-by-side comparison of all debug tiles: pre-warp analytical fill
and post-warp warped slot.